# CAD font matching lab

Test harness for step 5 (candidate graph construction) and step 6 (affine subgraph matching + step 6.5 scoring) of `docs/cad_font_vector_recognition.md`, scoped to **one hand-picked test candidate cluster** -- not the full multi-candidate pipeline (step 4, cross-candidate baseline-confidence state in 7/9-12 are out of scope; those need real clustering this harness doesn't build). One linear flow: load the character bank -> a gallery of every character's own graph -> the candidate's own raw vectors + graph -> match every character in the bank against the candidate (progress-tracked) -> render each character's top matches -> save stats -> a cross-character baseline-consensus overlay.

The character bank is built from a `scripts/label/CAD_font_label.py` output file -- this notebook is also the graph-construction diagnostic view for that data now (section 3), so there's no separate `cad_font_char_graph_lab.ipynb` any more. The test candidate is a **single labelled cluster from a `scripts/label/vector_label.py` output file** -- a controlled stand-in for step 4's real seqno-spatial clustering, so the matching math (transform fitting, correspondence search, scoring) can be visually validated in isolation before steps 4/7-12 are attempted.

See `docs/cad_font_matching_optimizations.md` for a literature survey (RANSAC, geometric hashing, pose clustering, ...) and a full complexity analysis of possible speedups to the matching step -- research only, not implemented here.

## 0 - Config

In [ ]:
from pathlib import Path

CAD_FONT_LABEL_JSON_PATH = None   # required: CAD_font_label.py output json (character bank source)
                                   # (default outputs/labels/<stem>_cad_font.json)
TEST_LABEL_JSON_PATH = None       # required: vector_label.py output json -- a single labelled
                                   # candidate cluster to test matching against
TEST_LABEL_ID = None              # None -> assume the test file has exactly one source="vector" entry

BEZIER_SAMPLE_COUNT = 5  # fixed points a "c" item is sampled into (2 endpoints + 3 generated)
                          # epsilon (connect/RDP tolerance) is no longer a config knob here -- it's
                          # fully data-derived per character (half the shortest post-split segment,
                          # see GraphBuildStats.epsilon, printed per character below)

OVERLAY_DPI = 600        # render_vector_cluster raster dpi for visualization -- high enough that
                          # individual lines/points stay legible at the large figsizes used below

N_GALLERY = None         # section 3 (character graph gallery): how many bank characters to render,
                          # complexity order -- None -> every template

# step 6.5 scoring weights (matching.score_match) -- tune here, not in matching.py
MSE_WEIGHT = 1.0
EDGE_WEIGHT = 1.0
EDGE_DEGREE_BLEND = 0.5
# per-point-type weights on the mse_term (original data vertex / true intersection-junction /
# select_anchor_points's synthesized baseline anchor) -- see matching.py::_critical_point_type
ORIGINAL_POINT_WEIGHT = 1.0
INTERSECTION_POINT_WEIGHT = 1.0
SYNTHETIC_POINT_WEIGHT = 1.0

TOP_K_PER_CHARACTER = 5   # sections 5/6: keep only the top-5 best matches PER character

OUTPUT_DIR = None        # None -> alongside TEST_LABEL_JSON_PATH, "<stem>_matches"


## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)


## 2 - Load the character bank

A `CAD_font_label.py` output JSON, re-extracting vectors per page from its own `pdf_path`. Section 3 below renders every one of these templates' own graphs -- this notebook is now the only place that visualization lives.

In [ ]:
from rastervec.Evaluation.Labelling.label_schema import load_labels
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.Evaluation.CadFont.character_bank import build_character_bank

assert CAD_FONT_LABEL_JSON_PATH, "set CAD_FONT_LABEL_JSON_PATH"
bank_labels = load_labels(CAD_FONT_LABEL_JSON_PATH)
assert bank_labels.pdf_path, f"{CAD_FONT_LABEL_JSON_PATH} has no pdf_path recorded"

cad_entries = [e for e in bank_labels.entries if e.source == "cad_font"]
print(f"{len(cad_entries)} cad_font character label(s), {len(bank_labels.baselines)} baseline(s)")

bank_reader = Reader(bank_labels.pdf_path)
bank_pages_needed = sorted({e.page_index for e in cad_entries})
bank_vectors_by_page = {p: extract_vectors(bank_reader.get_page(p)) for p in bank_pages_needed}
bank_entries_by_label_id = {e.label_id: e for e in cad_entries}
bank_baselines_by_id = {b.baseline_id: b for b in bank_labels.baselines}

templates = build_character_bank(
    bank_labels, bank_vectors_by_page,
    bezier_sample_count=BEZIER_SAMPLE_COUNT,
)
print(f"{len(templates)} character template(s) built (of {len(cad_entries)} cad_font entries)")


## 3 - Character graph gallery

Every `cad_font` character in the bank, complexity order (already how `build_character_bank` sorts `templates`), as a 2-panel figure: **left** = the character's own raw rendered vectors, no overlay; **right** = the same render with its graph overlaid -- category-colored critical points (**red** = endpoint, **orange** = junction, **magenta** = synthetic anchor; grey = non-critical) and its selected anchors starred + numbered. `N_GALLERY` caps how many templates get rendered (`None` = every one).

In [ ]:
import math
import matplotlib.pyplot as plt

from rastervec.Evaluation.CadFont.character_bank import to_baseline_relative_vectors
from rastervec.Evaluation.CadFont.graph import critical_point_indices
from rastervec.Evaluation.Labelling.label_schema import path_signature
from rastervec.commons.helpers.geometry import item_points, transform_point, union_bbox
from rastervec.commons.renderer.png import render_vector_cluster, page_points_to_pixel

_CATEGORY_COLOR = {"endpoint": "red", "junction": "orange", "synthetic": "magenta"}


def node_category(graph, i):
    if i in graph.synthetic_anchor_indices:
        return "synthetic"
    if graph.degrees()[i] > 2:
        return "junction"
    return "endpoint"


def draw_char_graph(ax, pixel_pts, edges, categories, anchor_indices=(), *,
                     edge_color="lime", default_color="lightgrey",
                     default_edgecolor="grey", default_size=15, category_size=40):
    """Shared node/edge drawing: `categories` maps a subset of node indices
    to one of `_CATEGORY_COLOR`'s keys -- anything not in `categories`
    draws as `default_color` (plain light-grey for the gallery/candidate
    graph plots below; the caller can instead pass an empty `categories` +
    a uniform `default_color` for a plain single-color overlay, e.g. the
    transformed-template overlay in `render_match`). `anchor_indices` (the
    SELECTED anchors, a subset of `categories`' keys, in slot order) get an
    extra black-outlined star marker plus a slot-number label on top of
    their existing color, so "what kind of point is this" and "which
    points did select_anchor_points actually pick" are both visible at
    once. Reused by the gallery (section 3), the candidate graph plot
    (section 4), and each rendered match's own panels (section 6)."""
    for a, b in edges:
        (x0, y0), (x1, y1) = pixel_pts[a], pixel_pts[b]
        ax.plot([x0, x1], [y0, y1], color=edge_color, linewidth=1.0, zorder=2)
    for i, (x, y) in enumerate(pixel_pts):
        cat = categories.get(i)
        if cat is not None:
            ax.scatter([x], [y], c=_CATEGORY_COLOR[cat], s=category_size, zorder=3, edgecolors="black", linewidths=0.5)
        else:
            ax.scatter([x], [y], c=default_color, s=default_size, zorder=2.5, edgecolors=default_edgecolor, linewidths=0.3)
    for slot, i in enumerate(anchor_indices):
        x, y = pixel_pts[i]
        ax.scatter(
            [x], [y], marker="*", s=260, zorder=6,
            facecolors="none", edgecolors="black", linewidths=1.5,
        )
        ax.annotate(
            str(slot + 1), (x, y), textcoords="offset points", xytext=(7, 7),
            fontsize=10, fontweight="bold", zorder=7,
        )


def inverse_baseline_transform_for(resolved_vectors, tvecs, baseline):
    """theta + translation to map a baseline-relative-frame point back to
    page space, solved from one known (page, baseline-relative) point pair."""
    theta = math.degrees(math.atan2(baseline.direction[1], baseline.direction[0]))
    p_page0 = item_points(resolved_vectors[0].items[0])[0]
    p_rel0 = item_points(tvecs[0].items[0])[0]
    r_rel0 = transform_point(p_rel0, offset=(0.0, 0.0), rotation_deg=theta)
    t_inv = (p_page0[0] - r_rel0[0], p_page0[1] - r_rel0[1])
    return theta, t_inv


def render_padding_for(resolved_vectors, page_pts, base_padding=2.0):
    """render_vector_cluster/page_points_to_pixel size their frame from the
    VECTORS' own bbox + a flat padding -- a synthesized anchor (or any
    other graph node) can legitimately land outside that bbox, so grow the
    padding by however far the given points overhang it on any side."""
    vec_bbox = union_bbox([v.bbox for v in resolved_vectors])
    xs = [p[0] for p in page_pts]
    ys = [p[1] for p in page_pts]
    overhang = max(
        vec_bbox[0] - min(xs), vec_bbox[1] - min(ys),
        max(xs) - vec_bbox[2], max(ys) - vec_bbox[3],
        0.0,
    )
    return base_padding + overhang


def build_template_panel(template):
    """Resolves `template`'s own page-space vectors (via its label entry +
    baseline) and renders its own graph at `OVERLAY_DPI` -- returns
    `(img, pixel_pts, categories, anchor_indices)`. Build ONCE per template
    and reuse across that template's own top-K rendered matches
    (`render_match`'s `template_panel` param) -- don't recompute this per
    match, it re-renders the template's own PDF geometry from scratch."""
    entry = bank_entries_by_label_id[template.label_id]
    baseline = bank_baselines_by_id[template.baseline_id]
    sig_map = {path_signature(v): v for v in bank_vectors_by_page[entry.page_index]}
    resolved = [sig_map[s] for s in entry.vector_signatures if s in sig_map]

    tvecs = to_baseline_relative_vectors(resolved, baseline.origin, baseline.direction)
    theta, t_inv = inverse_baseline_transform_for(resolved, tvecs, baseline)

    page_pts = [transform_point(n, offset=t_inv, rotation_deg=theta) for n in template.graph.nodes]
    padding = render_padding_for(resolved, page_pts)

    img = render_vector_cluster(resolved, dpi=OVERLAY_DPI, padding=padding)
    pixel_pts = page_points_to_pixel(resolved, OVERLAY_DPI, page_pts, padding=padding)

    critical = set(critical_point_indices(template.graph)) | set(template.graph.synthetic_anchor_indices)
    categories = {i: node_category(template.graph, i) for i in critical}
    return img, pixel_pts, categories, template.anchor_node_indices


gallery_templates = templates if N_GALLERY is None else templates[:N_GALLERY]
for template in gallery_templates:
    img, pixel_pts, categories, anchor_indices = build_template_panel(template)

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 6))
    ax_left.imshow(img)
    ax_left.set_title(f'"{template.text}" -- raw vectors')
    ax_left.axis("off")

    ax_right.imshow(img)
    draw_char_graph(ax_right, pixel_pts, template.graph.edges, categories, anchor_indices)
    ax_right.set_title(
        f'complexity={template.complexity:.0f}  nodes={template.graph.num_nodes()}  '
        f'edges={template.graph.num_edges()}  max_deg={template.graph.max_degree()}'
    )
    ax_right.axis("off")

    fig.suptitle(f'label_id={template.label_id}')
    plt.show()


## 4 - Candidate cluster: raw vectors + graph

Loads the single hand-picked candidate cluster (a `scripts/label/vector_label.py` output JSON, `source="vector"`) -- a controlled stand-in for step 4's real seqno-spatial clustering -- then builds its `CharGraph` (step 5: the same flatten/split/dedupe/connect/RDP pipeline a template gets, but skipping `to_baseline_relative_vectors` since no baseline is known yet for an unlabelled candidate). Rendered large, at the already-high `OVERLAY_DPI`, so individual nodes/edges stay legible without opening the saved image separately -- zoom via your browser/image viewer. **Left** = raw vectors (including bezier curves as PyMuPDF actually draws them); **right** = the same render with the candidate graph overlaid.

In [ ]:
from rastervec.Evaluation.CadFont.matching import build_candidate_graph

assert TEST_LABEL_JSON_PATH, "set TEST_LABEL_JSON_PATH"
test_labels = load_labels(TEST_LABEL_JSON_PATH)
assert test_labels.pdf_path, f"{TEST_LABEL_JSON_PATH} has no pdf_path recorded"

test_entries = [e for e in test_labels.entries if e.source == "vector"]
if TEST_LABEL_ID is not None:
    test_entry = next(e for e in test_entries if e.label_id == TEST_LABEL_ID)
else:
    assert len(test_entries) == 1, (
        f"expected exactly one source='vector' entry in {TEST_LABEL_JSON_PATH}, "
        f"found {len(test_entries)} -- set TEST_LABEL_ID to disambiguate"
    )
    test_entry = test_entries[0]

test_reader = Reader(test_labels.pdf_path)
test_page_vectors = extract_vectors(test_reader.get_page(test_entry.page_index))
test_sig_map = {path_signature(v): v for v in test_page_vectors}
resolved_test_vectors = [test_sig_map[s] for s in test_entry.vector_signatures if s in test_sig_map]
assert resolved_test_vectors, "no vector_signatures from the test label resolved against the test PDF"
print(f"test candidate cluster: {len(resolved_test_vectors)} vector(s), page {test_entry.page_index}")

candidate_graph, candidate_stats = build_candidate_graph(
    resolved_test_vectors, bezier_sample_count=BEZIER_SAMPLE_COUNT,
)
candidate_critical = critical_point_indices(candidate_graph)
print(
    f"candidate graph: nodes={candidate_graph.num_nodes()} edges={candidate_graph.num_edges()} "
    f"max_degree={candidate_graph.max_degree()} critical_points={len(candidate_critical)}  "
    f"epsilon={candidate_stats.epsilon:.4f}  "
    f"points removed by RDP: {candidate_stats.points_removed} "
    f"({candidate_stats.points_before_rdp} -> {candidate_stats.points_after_rdp})"
)

cand_img = render_vector_cluster(resolved_test_vectors, dpi=OVERLAY_DPI, padding=2.0)
cand_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, candidate_graph.nodes, padding=2.0)
cand_categories = {i: node_category(candidate_graph, i) for i in candidate_critical}

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(20, 10))
ax_left.imshow(cand_img)
ax_left.set_title(f"candidate -- raw vectors ({len(resolved_test_vectors)} vector(s))")
ax_left.axis("off")

ax_right.imshow(cand_img)
draw_char_graph(ax_right, cand_pixel_pts, candidate_graph.edges, cand_categories)
ax_right.set_title(
    f"candidate graph (page space) -- nodes={candidate_graph.num_nodes()} "
    f"edges={candidate_graph.num_edges()}"
)
ax_right.axis("off")
plt.show()


## 5 - Match every character against the candidate

`match_template_against_candidate` (step 6.3-6.5): enumerates every ordered, injective assignment of a template's first 2 anchors to the candidate graph's own critical points, filtered per-slot by "candidate degree >= that anchor's own degree" (`enumerate_anchor_correspondences`); fits a constrained similarity transform (uniform scale + rotation + translation -- no shear possible by construction) from each correspondence via a closed-form Umeyama least-squares fit; maps every template node into candidate space and nearest-neighbor-matches it against the candidate graph's *entire* node set; scores each result (step 6.5) specifically over the template's own critical points.

Iterates **every** `cad_font` character in the bank -- already sorted most-complex-first by `build_character_bank` (per step 6.1's own "test character graphs most-complex-first" design), with a `tqdm` progress bar across the whole bank. A template `passes_size_prefilter` rejects (structurally too small to ever match) is skipped with a one-line note. Each character keeps only its own top `TOP_K_PER_CHARACTER` matches by `score.total` (lower = better); one shared `candidate_tree` is built once and reused across every template. Pure matching only -- no rendering here, see section 6.

In [ ]:
from tqdm import tqdm

from rastervec.Evaluation.CadFont.matching import build_candidate_tree, match_template_against_candidate

candidate_tree = build_candidate_tree(candidate_graph)

all_top_matches = {}   # label_id -> list[CandidateMatch], top TOP_K_PER_CHARACTER, best first

for template in tqdm(templates, desc="matching characters"):
    stats = template.build_stats
    tmpl_matches = match_template_against_candidate(
        template, candidate_graph,
        mse_weight=MSE_WEIGHT, edge_weight=EDGE_WEIGHT, edge_degree_blend=EDGE_DEGREE_BLEND,
        original_point_weight=ORIGINAL_POINT_WEIGHT,
        intersection_point_weight=INTERSECTION_POINT_WEIGHT,
        synthetic_point_weight=SYNTHETIC_POINT_WEIGHT,
        candidate_tree=candidate_tree,
    )
    if not tmpl_matches:
        tqdm.write(
            f'"{template.text}" (label_id={template.label_id}) -- no possible match '
            f'(passes_size_prefilter rejected the candidate for this template)'
        )
        continue

    tmpl_matches = sorted(tmpl_matches, key=lambda m: m.score.total)[:TOP_K_PER_CHARACTER]
    all_top_matches[template.label_id] = tmpl_matches
    tqdm.write(
        f'"{template.text}" (label_id={template.label_id})  complexity={template.complexity:.0f}  '
        f'epsilon={stats.epsilon:.4f}  points removed by RDP: {stats.points_removed} '
        f'({stats.points_before_rdp} -> {stats.points_after_rdp})  '
        f'kept top {len(tmpl_matches)} match(es)'
    )

print(
    f"\n{sum(len(v) for v in all_top_matches.values())} total match(es) kept "
    f"across {len(all_top_matches)}/{len(templates)} character(s)"
)


## 6 - Top-K renders per character

Renders each character's kept top `TOP_K_PER_CHARACTER` matches from section 5, one `build_template_panel(template)` call per character (not per match, reused across that character's own top-K renders). Each match becomes a 3-panel figure: **1** = the template's own native graph, enlarged; **2** = a zoomed, un-annotated crop of the candidate's real vector art around just the matched region (the matched candidate nodes plus their directly-connected neighbors); **3** = the usual grey=unmatched / red-blue=matched candidate subgraph / green=transformed template (own anchors starred+numbered) / dashed green=fitted baseline, zoomed to that same region. Collects every kept match's fitted baseline for section 8's consensus overlay.

In [ ]:
from rastervec.commons.helpers.geometry import clip_line_to_bbox

if OUTPUT_DIR:
    OUTPUT_DIR_PATH = Path(OUTPUT_DIR)
else:
    OUTPUT_DIR_PATH = Path(TEST_LABEL_JSON_PATH).parent / (Path(TEST_LABEL_JSON_PATH).stem + "_matches")
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

_cand_bbox = union_bbox([v.bbox for v in resolved_test_vectors])
_x0, _y0, _x1, _y1 = _cand_bbox
PADDED_CAND_BBOX = (_x0 - 2.0, _y0 - 2.0, _x1 + 2.0, _y1 + 2.0)

templates_by_label_id = {t.label_id: t for t in templates}


def _match_crop_bbox_pixels(match, candidate_graph, cand_pixel_pts, *, margin_frac=0.25):
    """Pixel-space `(x0, x1, y0, y1)` covering the matched candidate nodes
    PLUS every node one hop out via a candidate edge -- "the portion of
    the candidate graph being matched to" -- expanded by a fractional
    margin (at least 10px) so the crop isn't razor-tight against the
    matched geometry."""
    matched = set(match.node_map.values())
    context = set(matched)
    for a, b in candidate_graph.edges:
        if a in matched:
            context.add(b)
        if b in matched:
            context.add(a)
    xs = [cand_pixel_pts[i][0] for i in context]
    ys = [cand_pixel_pts[i][1] for i in context]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    mx, my = max((x1 - x0) * margin_frac, 10.0), max((y1 - y0) * margin_frac, 10.0)
    return x0 - mx, x1 + mx, y0 - my, y1 + my


def render_match(match, idx, total, template_graph, template_anchor_indices, template_text, *,
                  template_panel, candidate_graph, cand_img, cand_pixel_pts, resolved_test_vectors,
                  padded_bbox, save_path=None):
    """Renders one match as 3 side-by-side panels, all at `OVERLAY_DPI`
    detail:

    1. The TEMPLATE's own native graph, enlarged (`template_panel`, from
       `build_template_panel` -- built ONCE per template by the caller and
       reused across that template's own top-K matches, not recomputed
       here) -- "an enlarged photo of the character graph".
    2. A zoomed, un-annotated crop of the candidate's real vector art
       around just the matched region (`_match_crop_bbox_pixels`: the
       matched candidate nodes plus their directly-connected neighbors) --
       "a crop of the original".
    3. The usual grey=unmatched / red-blue=matched candidate subgraph /
       green=entire transformed template (with its own selected anchors
       starred+numbered) / dashed green=fitted baseline overlay, zoomed to
       that same cropped region.

    The zoom in panels 2-3 is applied via `set_xlim`/`set_ylim` on the
    already-rendered, already-high-DPI `cand_img` -- no re-rendering or
    pixel-array cropping needed, since `cand_pixel_pts` are already valid
    pixel coordinates into that one image.

    Returns the clipped page-space baseline line segment (or `None`), so a
    caller can collect it for the cross-match consensus overlay (section
    8)."""
    matched_candidate_nodes = set(match.node_map.values())
    matched_candidate_edges = {
        (a, b) for a, b in candidate_graph.edges
        if a in matched_candidate_nodes and b in matched_candidate_nodes
    }
    crop_x0, crop_x1, crop_y0, crop_y1 = _match_crop_bbox_pixels(match, candidate_graph, cand_pixel_pts)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    tmpl_img, tmpl_pixel_pts, tmpl_categories, tmpl_anchor_indices = template_panel
    ax1.imshow(tmpl_img)
    draw_char_graph(ax1, tmpl_pixel_pts, template_graph.edges, tmpl_categories, tmpl_anchor_indices)
    ax1.set_title(f'"{template_text}" -- own graph')
    ax1.axis("off")

    ax2.imshow(cand_img)
    ax2.set_xlim(crop_x0, crop_x1)
    ax2.set_ylim(crop_y1, crop_y0)
    ax2.set_title("candidate region -- raw vectors")
    ax2.axis("off")

    ax3.imshow(cand_img)
    for a, b in candidate_graph.edges:
        if (a, b) in matched_candidate_edges:
            continue
        (x0, y0), (x1, y1) = cand_pixel_pts[a], cand_pixel_pts[b]
        ax3.plot([x0, x1], [y0, y1], color="grey", linewidth=0.8, zorder=1)
    for i, (x, y) in enumerate(cand_pixel_pts):
        if i in matched_candidate_nodes:
            continue
        ax3.scatter([x], [y], c="grey", s=15, zorder=1)

    for a, b in matched_candidate_edges:
        (x0, y0), (x1, y1) = cand_pixel_pts[a], cand_pixel_pts[b]
        ax3.plot([x0, x1], [y0, y1], color="blue", linewidth=1.5, zorder=2)
    for i in matched_candidate_nodes:
        x, y = cand_pixel_pts[i]
        ax3.scatter([x], [y], c="red", s=30, zorder=3, edgecolors="black", linewidths=0.5)

    transformed_page_pts = match.transform.apply_many(template_graph.nodes)
    transformed_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, transformed_page_pts, padding=2.0)
    draw_char_graph(
        ax3, transformed_pixel_pts, template_graph.edges, {}, template_anchor_indices,
        edge_color="green", default_color="green", default_edgecolor="black", default_size=25,
    )

    origin_page = match.transform.apply((0.0, 0.0))
    dir_probe = match.transform.apply((1.0, 0.0))
    dx, dy = dir_probe[0] - origin_page[0], dir_probe[1] - origin_page[1]
    length = math.hypot(dx, dy) or 1.0
    direction_page = (dx / length, dy / length)
    clipped = clip_line_to_bbox(origin_page, direction_page, padded_bbox)
    if clipped is not None:
        baseline_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, list(clipped), padding=2.0)
        (bx0, by0), (bx1, by1) = baseline_pixel_pts
        ax3.plot([bx0, bx1], [by0, by1], color="green", linestyle="--", linewidth=1.2, zorder=4)

    ax3.set_xlim(crop_x0, crop_x1)
    ax3.set_ylim(crop_y1, crop_y0)
    ax3.set_title(
        f'#{idx+1}/{total}  correspondence={match.correspondence}  '
        f'total={match.score.total:.4f}  mse={match.score.mse_term:.4f}  '
        f'edge={match.score.edge_term:.4f}  scale={match.transform.scale:.3f}  '
        f'rot={match.transform.rotation_deg:.1f}deg'
    )
    ax3.axis("off")

    fig.suptitle(f'"{template_text}"  match #{idx+1}/{total}')
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return clipped


all_baselines = []   # every kept match's clipped page-space baseline line, across every character

for label_id, tmpl_matches in all_top_matches.items():
    template = templates_by_label_id[label_id]
    tmpl_panel = build_template_panel(template)
    for idx, match in enumerate(tmpl_matches):
        clipped = render_match(
            match, idx, len(tmpl_matches), template.graph, template.anchor_node_indices, template.text,
            template_panel=tmpl_panel,
            candidate_graph=candidate_graph, cand_img=cand_img, cand_pixel_pts=cand_pixel_pts,
            resolved_test_vectors=resolved_test_vectors, padded_bbox=PADDED_CAND_BBOX,
            save_path=OUTPUT_DIR_PATH / f"{template.label_id}_match_{idx:03d}.png",
        )
        if clipped is not None:
            all_baselines.append(clipped)

print(f"{len(all_baselines)} match(es) rendered across {len(all_top_matches)}/{len(templates)} character(s)")


## 7 - Save full-bank debug stats

In [ ]:
import json

full_bank_rows = {
    label_id: {
        "text": next(t.text for t in templates if t.label_id == label_id),
        "build_stats": vars(next(t.build_stats for t in templates if t.label_id == label_id)),
        "matches": [
            {
                "index": idx,
                "correspondence": match.correspondence,
                "anchor_indices": match.anchor_indices,
                "scale": match.transform.scale,
                "rotation_deg": match.transform.rotation_deg,
                "translation": match.transform.translation,
                "score_total": match.score.total,
                "score_mse_term": match.score.mse_term,
                "score_edge_presence_cost": match.score.edge_presence_cost,
                "score_degree_cost": match.score.degree_cost,
            }
            for idx, match in enumerate(match_list)
        ],
    }
    for label_id, match_list in all_top_matches.items()
}
full_bank_stats_path = OUTPUT_DIR_PATH / "matches_full_bank.json"
with open(full_bank_stats_path, "w", encoding="utf-8") as f:
    json.dump(full_bank_rows, f, indent=2, default=str)
print(f"wrote {full_bank_stats_path}")


## 8 - Baseline consensus overlay

Every kept match's own fitted baseline (dashed green in each individual render above), from **every character tested in section 5**, drawn together on one shared raster of the candidate cluster at a low alpha -- overlapping/agreeing baselines from many different characters' matches stack into a visibly darker band, while one-off spurious matches stay faint.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(cand_img)

for p0, p1 in all_baselines:
    pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, [p0, p1], padding=2.0)
    (x0, y0), (x1, y1) = pixel_pts
    ax.plot([x0, x1], [y0, y1], color="green", linestyle="--", linewidth=1.5, alpha=0.15, zorder=4)

ax.set_title(f"baseline consensus -- {len(all_baselines)} match(es) across {len(all_top_matches)} character(s)")
ax.axis("off")
fig.savefig(OUTPUT_DIR_PATH / "baseline_consensus.png", dpi=150, bbox_inches="tight")
plt.show()
